# Задача B — Синтез нового вида/кадра для камерной установки

**Yandex ML Cup — трек ML**

**Цель:** восстановить, что одна из камер беспилотного автомобиля видела в промежуточный момент времени, имея соседние по времени кадры и плотное облако точек LiDAR.

**Метрика:** PSNR предсказанного изображения относительно эталона (больше — лучше) → переводится в скор соревнования.

---

## Постановка

У машины **установка из 6 камер**: `front`, `left_fwd`, `left_bwd`, `right_fwd`, `right_bwd`, `rear`.

Для каждого примера из `meta.json` даётся:
- Кадры со всех камер в момент **t0** и момент **t1** (разница 2 с)
- **Целевая камера** и **целевой момент** *между* t0 и t1
- Внутренние и внешние параметры камер (позы camera-to-world)
- **Плотное облако LiDAR** (~10 млн точек, 55 проходов) в мировых координатах

**Нужно предсказать изображение целевой камеры в целевой момент** — кадр, который никогда не записывался.

## Стратегия — смешиваем три независимые оценки

Ни один метод не надёжен везде, поэтому решение строит **три оценки** целевого кадра и смешивает их с весами доверия:

| Оценка | Используемый сигнал | Сильна там, где |
|---|---|---|
| **1. Временная интерполяция по оптическому потоку** | та же камера в t0 и t1 | плавное движение, почти статичная сцена |
| **2. Нейросетевая интерполяция RIFE** | кадры t0 и t1 (нейросеть) | нелинейное движение, деформация |
| **3. Рендеринг на основе LiDAR (IBR)** | 3D-геометрия + другие камеры | геометрия / параллакс, перекрытия |

Итоговые веса смешивания (подобраны): `temporal_strength=0.72`, `rife_mix=0.42`, `lidar_mix=0.30`.

## Оценка 1 — временная интерполяция по оптическому потоку

Цель находится на доле `α` пути от t0 к t1. Если знать, как каждый пиксель *движется* с кадра 0 на кадр 1, можно сдвинуть его на долю `α`.

1. Считаем плотный оптический поток **в обе стороны** через OpenCV **DIS**: `f01` (0→1) и `f10` (1→0).
2. Сдвигаем кадр 0 вперёд на `α·f01`, кадр 1 назад на `(1−α)·f10`.
3. Смешиваем два сдвига. Там, где они **расходятся** (перекрытие), доверяем стороне с меньшей ошибкой сдвига — мягкая маска перекрытий.

Поток вперёд-и-назад с маской расхождения — как раз то, что обрабатывает объекты, появляющиеся/исчезающие между кадрами.

In [ ]:
import cv2
import numpy as np

def flow_dis(a, b):
    """Dense optical flow a->b using OpenCV DIS (fast, robust)."""
    ga = cv2.cvtColor(a, cv2.COLOR_RGB2GRAY)
    gb = cv2.cvtColor(b, cv2.COLOR_RGB2GRAY)
    dis = cv2.DISOpticalFlow_create(cv2.DISOPTICAL_FLOW_PRESET_MEDIUM)
    return dis.calc(ga, gb, None)   # HxWx2 (dx, dy) per pixel

def remap_image(img, flow_xy, scale):
    """Warp img along a scaled flow field."""
    h, w = img.shape[:2]
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    map_x = xx + scale * flow_xy[..., 0]
    map_y = yy + scale * flow_xy[..., 1]
    return cv2.remap(img, map_x, map_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)

def temporal_interpolate(img0, img1, alpha):
    f01 = flow_dis(img0, img1)
    f10 = flow_dis(img1, img0)
    warp0 = remap_image(img0, f01, alpha)          # push frame 0 forward
    warp1 = remap_image(img1, f10, 1.0 - alpha)    # push frame 1 backward
    # occlusion-aware blend: weight toward the lower-error side
    err0 = np.abs(warp0.astype(np.float32) - img1).mean(-1, keepdims=True)
    err1 = np.abs(warp1.astype(np.float32) - img0).mean(-1, keepdims=True)
    w0 = (1 - alpha) * (err1 + 1e-3)
    w1 = alpha * (err0 + 1e-3)
    return ((w0 * warp0 + w1 * warp1) / (w0 + w1)).astype(np.uint8)

# temporal_interpolate(frame_t0, frame_t1, alpha=0.5)  -> predicted middle frame

## Оценка 2 — нейросетевая интерполяция RIFE

[RIFE](https://github.com/megvii-research/ECCV2022-RIFE) — нейросеть, созданная ровно для задачи «даны кадр 0 и кадр 1, синтезируй кадр в момент `α`». Она учит движение и перекрытия end-to-end и справляется с нелинейным, деформирующим движением, которое ручной оптический поток размазывает.

Её **подмешивают**, а не используют отдельно (`rife_mix=0.42`) — RIFE хороша в движении, но может «нафантазировать» текстуру, поэтому усреднение с оценкой по потоку сохраняет стабильность.

## Оценка 3 — рендеринг на основе LiDAR (путь геометрии)

Две оценки выше — чисто 2D. Но у нас есть **настоящая 3D-геометрия** из LiDAR — именно она чинит параллакс (близкие объекты сдвигаются сильнее дальних при движении машины).

Классический конвейер проективной геометрии:

1. **Проецируем** облако LiDAR в целевую камеру → плотная **карта глубины** целевого вида.
2. Для каждого целевого пикселя **обратно проецируем** его в 3D-точку мира по этой глубине.
3. **Перепроецируем** эту точку мира в *исходную* камеру (t0/t1, возможно другую камеру установки) и **берём её цвет** (билинейно).
4. Взвешиваем по доверию (насколько поверхность обращена к камере, насколько близко совпала глубина).

Проекция использует модель камеры-обскуры с дисторсией: $u = K \, [R \mid t]^{-1} X_{world}$.

In [ ]:
def world_to_camera(xyz, c2w):
    """World points -> camera frame using camera-to-world pose (invert it)."""
    R, t = c2w[:3, :3], c2w[:3, 3]
    return (xyz - t) @ R          # R is orthonormal, so R^{-1} == R^T

def project(xyz_world, c2w, K):
    """Project world points into pixel coords of a camera. Returns u, v, depth."""
    cam = world_to_camera(xyz_world, c2w)
    z = cam[:, 2]
    valid = z > 1e-3                          # keep points in front of the camera
    uv = (K @ cam.T).T
    u = uv[:, 0] / uv[:, 2]
    v = uv[:, 1] / uv[:, 2]
    return u, v, z, valid

def build_depth_map(lidar_xyz, target_c2w, target_K, h, w):
    """Splat lidar into the target view, keeping the nearest point per pixel (z-buffer)."""
    u, v, z, valid = project(lidar_xyz, target_c2w, target_K)
    depth = np.full((h, w), np.inf, np.float32)
    ui, vi = np.round(u).astype(int), np.round(v).astype(int)
    m = valid & (ui >= 0) & (ui < w) & (vi >= 0) & (vi < h)
    for uu, vv, zz in zip(ui[m], vi[m], z[m]):   # nearest wins (z-buffer)
        if zz < depth[vv, uu]:
            depth[vv, uu] = zz
    depth[np.isinf(depth)] = 0.0
    return depth

# In the real solution this is vectorized + densified (holes between sparse
# lidar returns are filled) before sampling colors from the source cameras.

## Собираем всё вместе

```
               ┌─────────────────────────────┐
  t0, t1  ───▶ │ 1. Интерп. DIS оптич. поток │──┐
  кадры        └─────────────────────────────┘  │
               ┌─────────────────────────────┐  │  взвешенное
  t0, t1  ───▶ │ 2. Нейроинтерполяция RIFE    │──┼──▶ смешивание ──▶ целевой кадр
  кадры        └─────────────────────────────┘  │  (по доверию)
               ┌─────────────────────────────┐  │
  lidar +  ──▶ │ 3. LiDAR-рендеринг (IBR)     │──┘
  позы         └─────────────────────────────┘
```

Всё считается в разрешении цели, смешивается по пикселям с весами доверия, затем сохраняется как JPEG высокого качества (`quality=95, subsampling=0`), потому что метрика — попиксельный PSNR.

## Почему смешивание, а не выбор одного?

PSNR наказывает любую сильно неверную область. Каждый метод ошибается по-своему:

- **Оптический поток** размазывает быстрое/нелинейное движение и тонкие структуры.
- **RIFE** может нафантазировать правдоподобную, но неверную текстуру.
- **LiDAR IBR** геометрически корректен, но имеет дыры (разреженные точки, перекрытия) и путает цвет под скользящими углами.

Взвешенное по доверию усреднение позволяет каждому методу закрывать провалы других — классическая причина, почему ансамбли выигрывают на метриках типа квадрата ошибки.

## Итог

- **Задача:** синтезировать незаписанный кадр камеры в промежуточный момент, метрика — PSNR.
- **Три взаимодополняющие оценки:** интерполяция по DIS-потоку, нейроинтерполяция RIFE и рендеринг на основе LiDAR.
- **Геометрия важна:** проекция плотного облака LiDAR в целевой вид и пересэмплирование исходных камер обрабатывают параллакс и перекрытия, недоступные чисто 2D-интерполяции.
- **Взвешенное по доверию смешивание** трёх оценок бьёт любой одиночный метод, потому что их режимы отказа не пересекаются.
- **Вывод с учётом метрики:** почти без потерь JPEG, потому что каждый пиксель важен для PSNR.